# 03 — Training Monitoring

Monitor training progress and compare model variants:
- Loss curves
- BLEU on dev set per epoch
- Comparison: BARTpho vs ViT5
- Single-task vs multi-task

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

RESULTS_DIR = Path("../results/checkpoints")

## Load Training Logs

HuggingFace Trainer saves `trainer_state.json` with training history.

In [ ]:
def load_training_log(checkpoint_dir):
    """Load trainer_state.json from a HF checkpoint."""
    state_path = Path(checkpoint_dir) / "trainer_state.json"
    if not state_path.exists():
        print(f"No trainer_state.json found at {state_path}")
        return None
    with open(state_path) as f:
        return json.load(f)

def extract_metrics(state):
    """Extract per-epoch metrics from trainer state."""
    if state is None:
        return []
    log_history = state.get("log_history", [])
    epochs = []
    for entry in log_history:
        if "eval_loss" in entry:
            epochs.append({
                "epoch": entry.get("epoch", 0),
                "eval_loss": entry.get("eval_loss", 0),
                "eval_bleu": entry.get("eval_bleu", 0),
                "train_loss": entry.get("loss", 0),
            })
    return epochs

# Load your checkpoint (update path as needed)
state = load_training_log(RESULTS_DIR)
if state:
    metrics = extract_metrics(state)
    print(f"Found {len(metrics)} epoch records")
else:
    print("No training log found yet. Run training first.")
    metrics = []

## Loss Curves

In [ ]:
if metrics:
    epochs = [m["epoch"] for m in metrics]
    eval_losses = [m["eval_loss"] for m in metrics]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, eval_losses, "o-", color="steelblue", label="Eval Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Progress — Eval Loss")
    ax.legend()
    plt.tight_layout()
    plt.savefig("../results/figures/training_loss.png", dpi=150)
    plt.show()
else:
    print("No metrics to plot.")

## BLEU on Dev Set

In [ ]:
if metrics and any(m.get("eval_bleu", 0) > 0 for m in metrics):
    eval_bleus = [m["eval_bleu"] for m in metrics]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, eval_bleus, "s-", color="coral", label="Eval BLEU")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("BLEU")
    ax.set_title("Training Progress — Dev BLEU")
    ax.legend()
    plt.tight_layout()
    plt.savefig("../results/figures/training_bleu.png", dpi=150)
    plt.show()
else:
    print("No BLEU metrics available yet.")

## Model Comparison Table

Fill in after running experiments with different models.

In [ ]:
# Template — fill in with actual results
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["BARTpho-syllable", "ViT5-base", "mBART-large-50"],
    "dialect2std BLEU": ["—", "—", "—"],
    "std2dialect BLEU": ["—", "—", "—"],
    "std2dialect DFR": ["—", "—", "—"],
    "lexnorm ERR": ["—", "—", "—"],
    "Training Time": ["—", "—", "—"],
})
print(comparison.to_markdown(index=False))